<div align="center">

# Week 02: Machine learning fundamentals

**Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists**<br>
Turkish Society of Cardiology · Prof. Dr. Utku Kose

[Week page](README.md) · [Lecture page](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html) · [Interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lab.html) · [PDF notes](Week02_Lecture_Notes.pdf) · [Open in Colab (fallback)](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/NB02_machine_learning_fundamentals.ipynb)

</div>

**How to use this Jupyter notebook.** Run the cells from top to bottom with Shift+Enter. Section 0 prepares the environment and must run first. Exercises print *correct* or *not yet* at once, and their reference solutions sit in collapsed cells: Open them only after trying. *Observe and reflect* cells ask open questions and have no solution cell. *Python note* cells explain the language where a new construct first appears. A participant who opens the notebook in the middle runs the *Catch-up* cell at the start of that section.

**Three paths.** All cells without a label form the **core path**, which is enough for everybody, including those who have never written code. Cells labelled **Advanced path** are optional extensions for those who already program. The **prompt path** reaches the same results by writing prompts instead of code. Every exercise has a collapsed *Prompt route* with a ready prompt for an artificial intelligence (AI) assistant, and every section ends with a *Prompt challenge*. Copy the prompt into an assistant, replace the content of the exercise cell with the code it returns and run it, and let the check decide. The [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md) of the week collects all prompts, and the [guide to the prompt path](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/PROMPT_PATH.md) explains which assistants work with this notebook.

**Where it runs.** The notebook is written for Jupyter in Anaconda on your own computer (see SETUP.md in the course folder). If installing Anaconda is not possible, the same notebook opens in Google Colab through the link at the top. Every data set here is synthetic or comes from a public research database. Real patient data belongs only on your own computer, under the rules of your institution, and never in a cloud notebook or in an artificial intelligence (AI) chat.

## 0. Setup

The first cell finds the course folder. If the notebook was opened on its own, for example in Google Colab, it downloads the course package `vclinic` and the data files from GitHub. The second cell loads the packages, chooses the language of the messages and checks the optional packages of the week.

In [ ]:
#@title Course files: run this cell (there is no need to open it)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
    "data/uci-heart-disease/processed.cleveland.data",
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # A downloaded copy, for example in Colab: the files of the package are refreshed.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Course files downloaded to", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Setup: packages, language and plotting style
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (adjusted_rand_score, average_precision_score,
                             confusion_matrix, precision_recall_curve,
                             roc_auc_score, roc_curve, silhouette_score)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     train_test_split)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from vclinic.datasets import load_cleveland
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 1, 0):
    raise ImportError("The course package vclinic is older than this notebook. Update the course folder (git pull, or download the ZIP file again) and restart the kernel.")
teach.set_language("en")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026

print("vclinic version", vclinic.__version__)
print("Course folder:", ROOT)
for name in ["ipywidgets", "lifelines"]:
    try:
        importlib.import_module(name)
        print(f"{name}: installed")
    except ImportError:
        print(f"{name}: not installed (optional; the notebook uses a built-in fallback)")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
teach.register_answers({
    "Exercise 2.1": (6, 0),
    "Exercise 2.2": (76, 0),
    "Exercise 2.3": (0.855, 0.01),
    "Exercise A": (0.62, 0.01),
    "Exercise B": (0.931, 0.005),
    "Exercise C": (0.8, 0.001),
    "Exercise D": (0.871, 0.005),
    "Exercise E": (0.496, 0.001),
    "Exercise F": (0.46, 0.03),
    "Exercise G": (0.309, 0.001),
    "Exercise H": (0.194, 0.01),
    "Exercise I": (0.733, 0.002),
    "Exercise J": (0.849, 0.01),
})
print("Expected answers are ready.")

## Python step 2: Data for models

A model of scikit-learn needs two things: a table of numbers with one row per patient, called `X`, and the outcome of each patient, called `y`. This step turns the Cleveland data of Week 1 into such a table. It selects columns and rows, handles missing values, turns a category into numbers, splits the patients into a training set and a test set, and fits a first model. Every later section of the notebook uses these steps.

In [ ]:
# Python practice: a small example for learning the language
heart = load_cleveland()
print(heart.shape)
heart[["age", "chol", "disease"]].head()

**Python note.** `heart.shape` gives the number of rows and columns. Double brackets, `heart[["age", "chol"]]`, select several columns and return a smaller DataFrame. Single brackets, `heart["age"]`, select one column and return a *Series*, a single column of values with the same row labels.

In [ ]:
# Python practice: a small example for learning the language
older = heart[heart["age"] >= 60]
print(len(older), "patients aged 60 or more")
heart.loc[heart["sex"] == 0, ["age", "chol", "disease"]].head(3)

**Python note.** `heart["age"] >= 60` compares every age with 60 and gives a column of `True` and `False`, a *Boolean mask*. Putting the mask in brackets keeps the rows where it is `True`. `.loc[rows, columns]` selects rows and columns at the same time: here the women (`sex == 0`) and three columns.

In [ ]:
# Python practice: a small example for learning the language
print(heart.isna().sum()[heart.isna().sum() > 0])
filled = heart["ca"].fillna(heart["ca"].median())
print("Missing in ca after filling:", filled.isna().sum())
print("Rows without any missing value:", len(heart.dropna()))

**Python note.** A missing value appears as `NaN` (not a number). `isna()` marks it, and `.sum()` counts the marks of each column. `fillna(value)` replaces missing values, here with the median of the column, and `dropna()` removes every row with a missing value. The variables of this week's model have no missing values, so the notebook keeps all 303 patients.

In [ ]:
# Python practice: a small example for learning the language
pd.get_dummies(heart["cp"], prefix="cp", drop_first=True, dtype=int).head()

**Python note.** The column `cp` codes the chest pain: 1 typical angina, 2 atypical angina, 3 non-anginal pain and 4 no pain. The numbers are names, not amounts: Pain of type 4 is not twice pain of type 2. `pd.get_dummies` therefore makes one *dummy variable* (0 or 1) per type. `drop_first=True` leaves out type 1, which becomes the reference: A patient with zeros in all three columns has typical angina.

In [ ]:
# The table of the model: one row per patient, numbers only
FEATURES = ["age", "sex", "trestbps", "chol", "thalach", "exang", "oldpeak", "cp"]
data = pd.get_dummies(heart[FEATURES + ["disease"]], columns=["cp"],
                      drop_first=True, dtype=int)
X = data.drop(columns="disease")
y = data["disease"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
print("X:", X.shape, " training set:", X_train.shape, " test set:", X_test.shape)
print("Share with disease: training", round(y_train.mean(), 3),
      " test", round(y_test.mean(), 3))

**Python note.** `train_test_split` returns four objects, and the line `X_train, X_test, y_train, y_test = ...` unpacks them into four names in this order. `test_size=0.25` keeps a quarter of the patients aside, `stratify=y` keeps the share of patients with disease the same in both parts, and `random_state` makes the split the same at every run. Here each patient has one row. When a patient has several rows, for example several ECGs, the split must keep all rows of a patient on the same side, which `GroupShuffleSplit` does with the patient identifier.

In [ ]:
# Python practice: a small example for learning the language
first = LogisticRegression(max_iter=1000)
first.fit(X_train, y_train)
print(first.predict(X_test[:5]))
print(first.predict_proba(X_test[:5]).round(3))

**Python note.** Every model of scikit-learn follows the same pattern. `fit(X, y)` learns from the training data. `predict(X)` returns a label, 0 or 1, and `predict_proba(X)` returns one column of probabilities per class: The first column is the probability of 0, the second of 1. `[:5]` takes the first five rows, and `max_iter=1000` lets the fitting run long enough.

In [ ]:
# Python practice: a small example for learning the language
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]
print("Probabilities of the first five test patients:", p_test[:5].round(3))

**Python note.** `make_pipeline` chains steps into one model. `StandardScaler` turns each column into a *z-score*, $(x - \bar{x})/s$, with the mean and the standard deviation of the training data. Inside the pipeline, `fit` learns these values from the training patients only, and the test patients are scaled with the same numbers, so nothing of the test set leaks into the model. `[:, 1]` takes all rows (`:`) and the second column (`1`) of a two-dimensional array.

**Quick check.** A model returns `predict_proba(X_test)` with shape `(76, 2)`. What do the 76 and the 2 mean?

<details><summary>Answer</summary>

76 rows are the 76 patients of the test set. The 2 columns are the probabilities of class 0 and class 1, which add up to 1 in every row.

</details>

**Exercise 2.1.** How many cells of `heart` are missing in total, over all rows and columns?

<details><summary><b>Prompt route</b> for Exercise 2.1: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: A pandas DataFrame named `heart` holds the Cleveland data of the UCI Heart Disease data set: 303 patients, one row per patient. The column disease is 0 or 1: 1 for coronary artery disease (CAD), 0 for none. Missing values are NaN.
Task: Count all missing cells of `heart`, over all rows and columns.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of missing cells as an integer. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 2.1", answer)`. Return only the code in one block, with a short comment above each step.
Test: First test the method on `pd.DataFrame({"a": [1, None], "b": [None, 4]})`, which must give 2. Then print the number of missing values of each column that has any; only the columns ca and thal may appear.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.1", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = int(heart.isna().sum().sum())
teach.check("Exercise 2.1", answer)

**Exercise 2.2.** How many patients go to the test set when the 303 patients are split with `test_size=0.25`? Answer with `len(...)`.

<details><summary><b>Prompt route</b> for Exercise 2.2: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `X`, a DataFrame with 303 rows and 10 numeric columns, and `y`, a Series with 303 values 0 or 1.
Task: Split `X` and `y` with `train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)` and count the patients of the test set.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The number of patients in the test set. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 2.2", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the sizes of both sets; they must add up to 303.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.2", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = len(X_test)
teach.check("Exercise 2.2", answer)

**Exercise 2.3.** What is the accuracy of `model`, the share of correctly labelled patients, on the test set? Use `model.score(X_test, y_test)` and round to three decimals.

<details><summary><b>Prompt route</b> for Exercise 2.3: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `X_train`, `X_test`, `y_train` and `y_test` come from `train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)`. `X` has the columns age (years), sex (1 man, 0 woman), trestbps (resting systolic blood pressure, mmHg), chol (serum cholesterol, mg/dL), thalach (maximum heart rate, bpm), exang (exercise angina, 0 or 1), oldpeak (ST depression, mm) and three dummy columns of the chest pain type, cp_2.0, cp_3.0 and cp_4.0. `y` is the column disease.
Task: Build `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`, fit it on the training set and compute its accuracy on the test set with `score`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The accuracy on the test set, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise 2.3", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print also the accuracy on the training set; it should be of the same size and not 1.0.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise 2.3", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(model.score(X_test, y_test), 3)
teach.check("Exercise 2.3", answer)

## 1. A first model: Logistic regression for coronary artery disease

*Lecture: [Logistic regression: From risk factors to a probability](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#logistic-regression-from-risk-factors-to-a-probability)*

The mini project of the session builds a model that estimates the probability of coronary artery disease (CAD) from seven variables and the type of chest pain of the 303 Cleveland patients. The model is fitted on the training set and studied in this section. Section 2 measures it on the test set.

**From the formulas to the code.** The linear predictor $z = \beta_0 + \sum_j \beta_j x_j$ uses `intercept_` for $\beta_0$ and `coef_` for the $\beta_j$. The probability $p = 1/(1 + e^{-z})$ is what `predict_proba` returns in its second column. The odds ratio of a variable is $e^{\beta_j}$, `np.exp(coef_)`. After `StandardScaler`, the $x_j$ are z-scores, so each odds ratio refers to one standard deviation (SD) of its variable.

**Started here?** Run the catch-up cell below once. It runs the code that Section 1 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(1)

In [ ]:
# Coefficients of the model as odds ratios per standard deviation
lr = model.named_steps["logisticregression"]
coefs = pd.DataFrame({"beta": lr.coef_[0], "odds ratio": np.exp(lr.coef_[0])},
                     index=X.columns).sort_values("beta")
print("Intercept beta_0 =", round(lr.intercept_[0], 3))
coefs.round(3)

**Python note.** `model.named_steps` holds the steps of a pipeline by name, so `named_steps["logisticregression"]` is the fitted logistic regression. `coef_[0]` is its first and only row of coefficients. `sort_values("beta")` orders the table from the most protective to the most harmful variable.

In [ ]:
# The case of the week on the treadmill: z and p by hand and by the model
case = pd.DataFrame([{"age": 57, "sex": 1, "trestbps": 119, "chol": 230, "thalach": 132,
                      "exang": 1, "oldpeak": 1.6, "cp_2.0": 1, "cp_3.0": 0, "cp_4.0": 0}])[X.columns]
scaler = model.named_steps["standardscaler"]
zscores = (case.to_numpy()[0] - scaler.mean_) / scaler.scale_
z = lr.intercept_[0] + np.sum(lr.coef_[0] * zscores)
p_hand = 1 / (1 + np.exp(-z))
print("z =", round(z, 3), " p (by hand) =", round(p_hand, 3),
      " p (model) =", round(model.predict_proba(case)[0, 1], 3))

**Python note.** `scaler.mean_` and `scaler.scale_` are the mean and the standard deviation that the scaler learned from the training set. `case.to_numpy()[0]` turns the one-row table into an array of ten numbers. `np.sum(a * b)` multiplies two arrays element by element and adds the products, which is the sum $\sum_j \beta_j x_j$ of the formula.

**Exercise A.** Patient VC-0246 has atypical angina (`cp_2.0 = 1`). What is his probability of CAD according to `model` if the chest pain is typical angina instead, with zeros in the three `cp` columns? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise A: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `model`, a fitted pipeline of StandardScaler and LogisticRegression for CAD, and `case`, a one-row DataFrame with the columns of `X`: age 57 years, sex 1, trestbps 119 mmHg, chol 230 mg/dL, thalach 132 bpm, exang 1, oldpeak 1.6 mm, cp_2.0 1, cp_3.0 0 and cp_4.0 0.
Task: Copy `case`, set cp_2.0 to 0 so that all three chest pain columns are 0 (typical angina), and compute the probability of CAD with `model.predict_proba`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The probability, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise A", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the probability of the unchanged `case` as well; the two probabilities must differ and both lie between 0 and 1.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise A", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
typical = case.copy()
typical["cp_2.0"] = 0
answer = round(model.predict_proba(typical)[0, 1], 3)
teach.check("Exercise A", answer)

**Observe and reflect.** In the Cleveland data, the patients without chest pain (`cp_4.0 = 1`) have the highest odds of disease, and typical angina, the reference, has lower odds. How can a referral pattern explain this? What does it mean for using the model in a clinic that sees a different mix of patients?

**Advanced path.** Fit the same pipeline with `LogisticRegression(C=0.05, max_iter=1000)`. A small `C` means strong *regularisation*: The coefficients shrink toward zero. Compare the two tables of odds ratios and the test AUC of Section 2.

In [ ]:
# Advanced path
strong = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=1000)).fit(X_train, y_train)
pd.DataFrame({"C=1": np.exp(lr.coef_[0]),
              "C=0.05": np.exp(strong.named_steps["logisticregression"].coef_[0])},
             index=X.columns).round(2)

**Prompt challenge, Section 1: A forest plot of the odds ratios.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `model`, a fitted pipeline of StandardScaler and LogisticRegression for CAD, and `X`, whose columns are the variables of the model.
Task: Draw the odds ratio per standard deviation of each variable as a point on a logarithmic horizontal axis, with a vertical line at 1 and the variable names on the vertical axis.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure. Return only the code in one block, with a short comment above each step.
Test: Print the odds ratios as well; thalach must lie below 1 and oldpeak above 1.
</pre>

</details>

**What to check.** The odds ratio per SD is about 0.58 for thalach and about 2.07 for oldpeak; sex and cp_4.0 are above 2.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 2. Measuring a model: Confusion matrix and ROC curve

*Lecture: [Measuring a model: Confusion matrix and ROC curve](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#measuring-a-model-confusion-matrix-and-roc-curve)*

The test set holds 76 patients that the model has never seen. A threshold turns each probability into a decision, and the decisions are counted against the truth in a *confusion matrix*. Moving the threshold trades sensitivity against specificity, and the *receiver operating characteristic (ROC) curve* shows all thresholds at once.

**From the formulas to the code.** With the counts `tp`, `fp`, `tn` and `fn`, sensitivity is $Se = TP/(TP+FN)$, specificity $Sp = TN/(TN+FP)$, the positive predictive value $PPV = TP/(TP+FP)$ and the negative predictive value $NPV = TN/(TN+FN)$. `roc_curve` returns the false positive rate $1 - Sp$, the true positive rate $Se$ and the thresholds, and `roc_auc_score` the area under the curve (AUC). Youden's index is $J = Se + Sp - 1$.

**Started here?** Run the catch-up cell below once. It runs the code that Section 2 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(2)

In [ ]:
# Confusion matrix at the threshold 0.5
pred = (p_test >= 0.5).astype(int)
print(pd.crosstab(y_test.to_numpy(), pred, rownames=["truth"], colnames=["model"]))
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"TP={tp} FP={fp} TN={tn} FN={fn}")
print("Se =", round(tp / (tp + fn), 3), " Sp =", round(tn / (tn + fp), 3),
      " PPV =", round(tp / (tp + fp), 3), " NPV =", round(tn / (tn + fn), 3))

**Python note.** `(p_test >= 0.5)` gives `True` or `False` for each patient, and `.astype(int)` turns them into 1 and 0. `pd.crosstab` counts the pairs of truth and decision. `confusion_matrix(...).ravel()` returns the four counts in the order TN, FP, FN, TP, and the line unpacks them into four names. An *f-string* puts values inside braces into the text.

In [ ]:
# ROC curve, AUC and Youden's index
fpr, tpr, thresholds = roc_curve(y_test, p_test)
auc = roc_auc_score(y_test, p_test)
best = np.argmax(tpr - fpr)
print("AUC =", round(auc, 3))
print("Youden: threshold", round(thresholds[best], 3), " Se", round(tpr[best], 3),
      " Sp", round(1 - fpr[best], 3), " J", round(tpr[best] - fpr[best], 3))
fig, ax = plt.subplots(figsize=(4.8, 4.6))
ax.plot(fpr, tpr, color="#9B1B1B", lw=2, label=f"logistic regression (AUC {auc:.3f})")
ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1, label="chance")
ax.scatter(fpr[best], tpr[best], color="#1F5F8B", zorder=3, label="Youden")
ax.set_xlabel("1 − specificity (false positive rate)")
ax.set_ylabel("Sensitivity (true positive rate)")
ax.legend(loc="lower right")
plt.show()

**Python note.** `roc_curve` tries every useful threshold and returns three arrays of the same length. `np.argmax(tpr - fpr)` gives the position of the largest value of $Se - (1 - Sp) = J$, and that position picks the matching threshold.

**Explore.** Move the threshold. The confusion matrix, the four measures and the point on the ROC curve change together. A low threshold suits a test that should miss nobody, a high one a test that should start an invasive procedure only when it is fairly sure.

In [ ]:
# Explore: the threshold
def show_threshold(threshold=0.5):
    pred = (p_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    se, sp = tp / (tp + fn), tn / (tn + fp)
    ppv = tp / (tp + fp) if tp + fp else float("nan")
    fig, ax = plt.subplots(figsize=(4.4, 4.2))
    ax.plot(fpr, tpr, color="#9B1B1B", lw=2)
    ax.plot([0, 1], [0, 1], color="#888888", ls="--", lw=1)
    ax.scatter(1 - sp, se, s=60, color="#1F5F8B", zorder=3)
    ax.set_xlabel("1 − Sp")
    ax.set_ylabel("Se")
    ax.set_title(f"threshold {threshold:.2f}")
    plt.show()
    print(f"TP={tp} FP={fp} TN={tn} FN={fn} | Se={se:.3f} Sp={sp:.3f} PPV={ppv:.3f}")


try:
    from ipywidgets import interact, FloatSlider
    interact(show_threshold, threshold=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05,
                                                   description="threshold"))
except ImportError:
    for value in (0.3, 0.5, 0.7):
        show_threshold(value)

**Exercise B.** What is the AUC of `model` on the test set? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise B: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `model`, a fitted pipeline for CAD, and the test set `X_test`, `y_test` (76 patients; `y_test` is 0 or 1, with 1 for CAD).
Task: Compute the area under the receiver operating characteristic curve (AUC) of the probabilities of `model` on the test set with `roc_auc_score`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The AUC, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise B", answer)`. Return only the code in one block, with a short comment above each step.
Test: Compute also the AUC of 1 minus the probabilities; the two values must add up to 1.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise B", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3)
teach.check("Exercise B", answer)

**Exercise C.** What is the sensitivity of `model` on the test set at the threshold 0.5? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise C: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `model`, a fitted pipeline for CAD, and the test set `X_test`, `y_test` (76 patients, 35 of them with CAD; `y_test` is 0 or 1).
Task: Label a patient positive when the probability of CAD is 0.5 or more, count TP, FP, TN and FN with `confusion_matrix(...).ravel()` and compute the sensitivity TP / (TP + FN).
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The sensitivity, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise C", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print TP and FN; their sum must be 35, the number of test patients with CAD.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise C", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
pred = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
answer = round(tp / (tp + fn), 3)
teach.check("Exercise C", answer)

**Observe and reflect.** An AUC of 0.93 sounds excellent. Which questions should a cardiologist still ask before using the model for decisions about angiography? Think of the size of the test set, of where the patients came from, and of what a false negative costs compared with a false positive.

**Prompt challenge, Section 2: Three thresholds side by side.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `p_test`, the probabilities of CAD of the 76 test patients, and `y_test`, their outcome.
Task: Build a table with one row for each of the thresholds 0.3, 0.5 and 0.7 and the columns Se, Sp, PPV and NPV.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: A pandas DataFrame rounded to three decimals. Return only the code in one block, with a short comment above each step.
Test: At 0.5 the table must show Se 0.800 and Sp 0.902, as in Section 2.
</pre>

</details>

**What to check.** Sensitivity falls from 0.857 to 0.571 and specificity rises from 0.805 to 0.951 between the thresholds 0.3 and 0.7.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 3. Cross-validation and data leakage

*Lecture: [Training and testing: Cross-validation and data leakage](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#training-and-testing-cross-validation-and-data-leakage)*

One test set of 76 patients gives one number, and another split would give another. *K-fold cross-validation* splits the patients into $K$ parts, trains on $K-1$ of them, tests on the remaining one and repeats this $K$ times. The mean of the $K$ results is a more stable estimate, and their spread shows how much it moves. Every step that learns from the data, scaling and selection of variables included, must be repeated inside each fold, or information from the test patients *leaks* into the model.

**From the formulas to the code.** The estimate is $\overline{AUC} = \frac{1}{K}\sum_{k=1}^{K} AUC_k$. `StratifiedKFold(5, shuffle=True, random_state=...)` makes the five folds with the same share of disease in each, and `cross_val_score` returns the five $AUC_k$.

**Started here?** Run the catch-up cell below once. It runs the code that Section 3 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(3)

In [ ]:
# Five-fold cross-validation of the whole pipeline
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
print("AUC of the five folds:", scores.round(3))
print("Mean", round(scores.mean(), 3), " SD", round(scores.std(), 3))

**Python note.** `cross_val_score` takes the unfitted pipeline and the whole `X` and `y`. In each fold it fits a fresh copy of the pipeline on four parts, so the scaler also learns from the training folds only, and it scores the fifth part. `scoring="roc_auc"` asks for the AUC.

In [ ]:
# Leakage on pure noise: 500 random variables and no real signal
rng = np.random.default_rng(RANDOM_STATE)
noise = rng.normal(size=(len(y), 500))

# Wrong: choose the 10 "best" variables on all patients, then cross-validate
chosen = SelectKBest(f_classif, k=10).fit_transform(noise, y)
leaky = cross_val_score(LogisticRegression(max_iter=1000), chosen, y, cv=cv, scoring="roc_auc")

# Right: the selection is a step of the pipeline and is repeated in each fold
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=1000)),
                         noise, y, cv=cv, scoring="roc_auc")
print("Selection before cross-validation: AUC", round(leaky.mean(), 3))
print("Selection inside the pipeline:     AUC", round(honest.mean(), 3))

**Python note.** `rng.normal(size=(303, 500))` draws a table of random numbers with 303 rows and 500 columns. `SelectKBest(f_classif, k=10)` keeps the 10 columns that differ most between the two groups. On pure noise some columns differ by chance, and choosing them with all patients lets the test folds take part in the choice.

**Exercise D.** What is the mean cross-validated AUC of `pipe` over the five folds of `cv`? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise D: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `X` and `y`, the 303 Cleveland patients (`y` is 0 or 1, with 1 for CAD), and `pipe`, an unfitted pipeline of StandardScaler and LogisticRegression(max_iter=1000).
Task: Run a five-fold cross-validation with `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)` and `cross_val_score(..., scoring="roc_auc")`, and take the mean of the five AUC values.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The mean AUC, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise D", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the five AUC values; each must lie between 0.5 and 1.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise D", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean(), 3)
teach.check("Exercise D", answer)

**Observe and reflect.** The leaky version finds an AUC of about 0.7 in data that contain no information at all. Published models are sometimes built this way. Which sentence in the methods section of a paper would make you suspect it, and which sentence would reassure you?

**Advanced path.** Repeat the cross-validation ten times with different shuffles, using `RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)` from `sklearn.model_selection`, and plot the histogram of the 50 AUC values.

In [ ]:
# Advanced path
from sklearn.model_selection import RepeatedStratifiedKFold
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
many = cross_val_score(pipe, X, y, cv=rcv, scoring="roc_auc")
plt.hist(many, bins=12, color="#1F5F8B")
plt.xlabel("AUC")
plt.ylabel("Number of folds")
plt.show()
print(round(many.mean(), 3), round(many.std(), 3))

**Prompt challenge, Section 3: Logistic regression or random forest?.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `X` and `y`, the 303 Cleveland patients, and the folds `cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)`.
Task: Compare the mean cross-validated AUC of the pipeline of StandardScaler and LogisticRegression(max_iter=1000) with that of `RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)` on the same folds.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The two means and standard deviations. Return only the code in one block, with a short comment above each step.
Test: The logistic regression must give the mean AUC of Exercise D.
</pre>

</details>

**What to check.** Both models reach a mean AUC of about 0.87; the random forest is not better on these 303 patients.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 4. Decision trees and random forests

*Lecture: [Decision trees and random forests](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#decision-trees-and-random-forests)*

A decision tree asks one yes or no question at a time, like a clinical algorithm. At each node it chooses the question that makes the two groups purest. A random forest grows hundreds of trees, each on a bootstrap sample of the patients and with a random subset of the variables at each node, and averages their probabilities.

**From the formulas to the code.** The Gini impurity of a node with the class shares $p_0$ and $p_1$ is $G = 1 - (p_0^2 + p_1^2)$. A split is scored by the impurity of its two children, weighted by their sizes. `DecisionTreeClassifier(max_depth=3)` limits the tree to three levels of questions, and `RandomForestClassifier(n_estimators=300)` averages 300 trees.

**Started here?** Run the catch-up cell below once. It runs the code that Section 4 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(4)

In [ ]:
# A tree with three levels of questions
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Test AUC of the tree:", round(roc_auc_score(y_test, tree.predict_proba(X_test)[:, 1]), 3))
fig, ax = plt.subplots(figsize=(13, 5.5))
plot_tree(tree, feature_names=list(X.columns), class_names=["no CAD", "CAD"],
          filled=True, impurity=True, fontsize=8, ax=ax)
plt.show()

**Python note.** `plot_tree` draws each node with its question, its Gini impurity, the number of patients (`samples`) and the count of each class (`value`). The left branch of a node holds the patients for whom the question is true.

**Exercise E.** Compute the Gini impurity of the root of the tree, before any question, from the counts of the two classes in `y_train`. Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise E: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `y_train`, the outcome of the 227 training patients, 0 or 1.
Task: Write a function `gini(n0, n1)` that returns 1 - (p0² + p1²) with p0 = n0 / (n0 + n1) and p1 = n1 / (n0 + n1), and apply it to the counts of 0 and 1 in `y_train`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The Gini impurity, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise E", answer)`. Return only the code in one block, with a short comment above each step.
Test: Check with assert statements that gini(50, 50) is 0.5 and gini(100, 0) is 0.0.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise E", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
p0, p1 = n0 / (n0 + n1), n1 / (n0 + n1)
answer = round(1 - (p0 ** 2 + p1 ** 2), 3)
teach.check("Exercise E", answer)

In [ ]:
# A random forest and the permutation importance of its variables
forest = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
print("Test AUC of the forest:", round(roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1]), 3))
imp = permutation_importance(forest, X_test, y_test, scoring="roc_auc", n_repeats=20,
                             random_state=RANDOM_STATE)
order = np.argsort(imp.importances_mean)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(X.columns[order], imp.importances_mean[order], xerr=imp.importances_std[order], color="#1F5F8B")
ax.set_xlabel("Drop in AUC when the variable is shuffled")
plt.show()

**Python note.** *Permutation importance* shuffles one column of the test set at a time and measures how much the AUC drops. A large drop means that the forest relies on that variable. `np.argsort` returns the positions that would sort the values, so the bars appear from the smallest to the largest.

**Observe and reflect.** The single tree is easy to read but reaches a lower AUC than the forest, and the forest is accurate but hard to read. For a decision at the bedside, which would you prefer, and what would you need to see before trusting the forest?

**Prompt challenge, Section 4: How deep should a tree be?.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `X_train`, `X_test`, `y_train` and `y_test` come from `train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)`. `X` has the columns age (years), sex (1 man, 0 woman), trestbps (resting systolic blood pressure, mmHg), chol (serum cholesterol, mg/dL), thalach (maximum heart rate, bpm), exang (exercise angina, 0 or 1), oldpeak (ST depression, mm) and three dummy columns of the chest pain type, cp_2.0, cp_3.0 and cp_4.0. `y` is the column disease.
Task: Fit `DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)` for d from 1 to 8 on the training set and plot the test AUC against the depth.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and the eight AUC values. Return only the code in one block, with a short comment above each step.
Test: The depth 3 must give the test AUC of the tree of Section 4.
</pre>

</details>

**What to check.** The AUC peaks at a depth of 3, about 0.81, and falls below 0.75 for deeper trees: a picture of overfitting.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 5. The risk outpatient clinic: Coronary artery disease in the registry

*Lecture: [The virtual clinic, stage 2: The risk outpatient clinic opens](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#the-virtual-clinic-stage-2-the-risk-outpatient-clinic-opens)*

The virtual clinic grows to 2000 patients and opens its risk outpatient clinic. The course generated the coronary artery disease of each patient from known log-odds effects, stored in `vclinic.TRUE_EFFECTS`. A logistic regression fitted to the registry should therefore find coefficients close to these values, which a model on real data can never show. `vclinic.add_followup` adds the echocardiographic details and the follow-up used from Section 6 on.

**From the formulas to the code.** The variables are coded as in `TRUE_EFFECTS`: age per 10 years above 60, $(LDL - 130)/40$ and $(HDL - 50)/15$, and 0 or 1 for the others. `LogisticRegression(C=np.inf)` fits the model without regularisation, so its `coef_` are the maximum likelihood estimates of the $\beta_j$.

**Started here?** Run the catch-up cell below once. It runs the code that Section 5 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(5)

In [ ]:
# The registry with follow-up: 2000 patients
registry = vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))
print(registry.shape)
print(registry["main_diagnosis"].value_counts().to_string())


def risk_table(df):
    """The variables coded as in vclinic.TRUE_EFFECTS."""
    return pd.DataFrame({
        "age_per_10y": (df["age"] - 60) / 10,
        "male": (df["sex"] == "M").astype(int),
        "smoker": df["smoker"],
        "diabetes": df["diabetes"],
        "hypertension": df["hypertension"],
        "ldl_per_40": (df["ldl"] - 130) / 40,
        "hdl_per_15": (df["hdl"] - 50) / 15,
        "family_history": df["family_history"],
    })


Z = risk_table(registry)
cad_model = LogisticRegression(C=np.inf, max_iter=2000).fit(Z, registry["cad"])
truth = pd.Series(vclinic.TRUE_EFFECTS)
compare = pd.DataFrame({"learned": np.r_[cad_model.intercept_, cad_model.coef_[0]],
                        "true": truth[["intercept"] + list(Z.columns)].to_numpy()},
                       index=["intercept"] + list(Z.columns))
compare.round(2)

**Python note.** A *docstring*, the text in triple quotes right under `def`, describes what a function does. `np.r_[a, b]` joins two arrays into one. `truth[["intercept"] + list(Z.columns)]` puts the true values in the same order as the learned ones, so that the two columns of the table can be compared row by row.

In [ ]:
# The case of the week, VC-0246, in the risk outpatient clinic
row = registry[registry["patient_id"] == "VC-0246"]
z_true = truth["intercept"] + (risk_table(row).iloc[0] * truth[Z.columns]).sum()
print(row[["age", "sex", "smoker", "hypertension", "diabetes", "ldl", "hdl", "family_history"]].to_string(index=False))
print("Probability of CAD, model:", round(cad_model.predict_proba(risk_table(row))[0, 1], 3),
      " true:", round(1 / (1 + np.exp(-z_true)), 3))
print("Coronary angiography in the registry (cad):", int(row["cad"].iloc[0]))

**Exercise F.** What log-odds coefficient does `cad_model` learn for `smoker`? Round to two decimals and compare it with `TRUE_EFFECTS["smoker"]`.

<details><summary><b>Prompt route</b> for Exercise F: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: A pandas DataFrame named `registry` holds 2000 synthetic patients of the virtual clinic, one row per patient, made with `vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))`. Among its columns are age (years), sex ("M" or "F"), smoker, diabetes, hypertension, family_history and cad (0 or 1), ldl and hdl (mg/dL), egfr (mL/min/1.73 m²), lvef (%), ntprobnp (pg/mL), af (0 or 1), bmi (kg/m²), sbp (mmHg), main_diagnosis, followup_months (months) and event (1 for death or hospitalisation for heart failure, 0 for censored).
Task: Code the variables as in `vclinic.TRUE_EFFECTS`: age_per_10y = (age - 60) / 10, male = 1 for sex "M", smoker, diabetes, hypertension, ldl_per_40 = (ldl - 130) / 40, hdl_per_15 = (hdl - 50) / 15 and family_history. Fit `LogisticRegression(C=np.inf, max_iter=2000)` to the column cad and read the coefficient of smoker.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The coefficient of smoker, rounded to two decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise F", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print all learned coefficients next to the values of `vclinic.TRUE_EFFECTS`; each should lie within about 0.2 of its true value.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise F", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(cad_model.coef_[0][list(Z.columns).index("smoker")], 2)
teach.check("Exercise F", answer)

**Observe and reflect.** The learned coefficients are close to the truth but not equal to it, although the model is the right one. What would happen to the differences with 200 patients, and with 20000? VC-0246 has a probability of about 0.8 and a normal angiogram. Is the model wrong for him?

**Prompt challenge, Section 5: The risk of the case across the ages.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: A pandas DataFrame named `registry` holds 2000 synthetic patients of the virtual clinic, one row per patient, made with `vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))`. Among its columns are age (years), sex ("M" or "F"), smoker, diabetes, hypertension, family_history and cad (0 or 1), ldl and hdl (mg/dL), egfr (mL/min/1.73 m²), lvef (%), ntprobnp (pg/mL), af (0 or 1), bmi (kg/m²), sbp (mmHg), main_diagnosis, followup_months (months) and event (1 for death or hospitalisation for heart failure, 0 for censored). `cad_model` is a fitted LogisticRegression on the table `risk_table(registry)`, and the function `risk_table(df)` codes the variables as in `vclinic.TRUE_EFFECTS`.
Task: Take the row of patient VC-0246, change only the age from 40 to 80 years in steps of one year and plot the probability of CAD of `cad_model` against the age.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure. Return only the code in one block, with a short comment above each step.
Test: At 57 years the curve must pass through the probability that Section 5 prints for VC-0246.
</pre>

</details>

**What to check.** The curve rises from about 0.55 at 40 years to about 0.9 at 80 years and passes through about 0.76 at 57 years.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 6. Rare outcomes: Imbalanced data

*Lecture: [Rare outcomes: Imbalanced data](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#rare-outcomes-imbalanced-data)*

Death or hospitalisation for heart failure within 12 months happens to about 3 of 100 patients of the registry. With such a rare outcome, a model that always answers *no event* is right 97 times out of 100 and useless. This section shows why accuracy misleads, how class weights shift a model toward the rare class and why the positive predictive value depends on the prevalence.

**From the formulas to the code.** Bayes' theorem gives $PPV = \frac{Se \cdot \pi}{Se \cdot \pi + (1 - Sp)(1 - \pi)}$ for a prevalence $\pi$. `class_weight="balanced"` gives each class the weight $n/(2 n_k)$, so that the rare class counts as much as the common one. `precision_recall_curve` returns the PPV (*precision*) and the sensitivity (*recall*) at every threshold, and `average_precision_score` summarises the curve.

**Started here?** Run the catch-up cell below once. It runs the code that Section 6 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(6)

In [ ]:
# The rare outcome: an event within 12 months
registry["event_12m"] = ((registry["event"] == 1) & (registry["followup_months"] <= 12)).astype(int)
EVENT_FEATURES = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes"]
Xe = registry[EVENT_FEATURES].assign(ntprobnp=np.log(registry["ntprobnp"]))
ye = registry["event_12m"]
Xe_train, Xe_test, ye_train, ye_test = train_test_split(
    Xe, ye, test_size=0.3, stratify=ye, random_state=RANDOM_STATE)
print("Prevalence:", round(ye.mean(), 3), " events in the test set:", int(ye_test.sum()), "/", len(ye_test))
print("Accuracy of always answering no event:", round((ye_test == 0).mean(), 3))

rows = []
for weight in [None, "balanced"]:
    m = make_pipeline(StandardScaler(), LogisticRegression(class_weight=weight, max_iter=1000)).fit(Xe_train, ye_train)
    p = m.predict_proba(Xe_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(ye_test, (p >= 0.5).astype(int)).ravel()
    rows.append({"class_weight": str(weight), "AUC": roc_auc_score(ye_test, p),
                 "AP": average_precision_score(ye_test, p), "accuracy": (tp + tn) / len(ye_test),
                 "Se": tp / (tp + fn), "PPV": tp / (tp + fp) if tp + fp else np.nan})
pd.DataFrame(rows).round(3)

**Python note.** `.assign(ntprobnp=...)` returns a copy of the table in which one column is replaced, here by its logarithm, because NT-proBNP spans several orders of magnitude. `str(weight)` turns `None` into the text "None" for the table. `np.nan` marks a value that cannot be computed, a PPV without any positive decision.

In [ ]:
# Precision and recall, and the PPV as a function of the prevalence
balanced = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(Xe_train, ye_train)
precision, recall, _ = precision_recall_curve(ye_test, balanced.predict_proba(Xe_test)[:, 1])
prev = np.linspace(0.005, 0.5, 200)
se, sp = 0.85, 0.90
ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(recall, precision, color="#9B1B1B")
ax[0].axhline(ye_test.mean(), color="#888888", ls="--", label="prevalence")
ax[0].set_xlabel("Recall (sensitivity)")
ax[0].set_ylabel("Precision (PPV)")
ax[0].legend()
ax[1].plot(prev * 100, ppv, color="#1F5F8B")
ax[1].set_xlabel("Prevalence (%)")
ax[1].set_ylabel("PPV")
ax[1].set_title("Se 0.85, Sp 0.90")
plt.show()

**Python note.** `np.linspace(0.005, 0.5, 200)` makes 200 evenly spaced prevalences. The formula of Bayes' theorem then runs on all of them at once, because NumPy works element by element on arrays. `_` is a name for a returned value that is not needed, here the thresholds.

**Exercise G.** A screening model has a sensitivity of 0.85 and a specificity of 0.90. What is its PPV in a population with a prevalence of 5 percent? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise G: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: A screening model with a sensitivity of 0.85 and a specificity of 0.90, used in a population with a prevalence of 0.05 (5%).
Task: Write a function `ppv(se, sp, prev)` with Bayes' theorem, PPV = se × prev / (se × prev + (1 - sp) × (1 - prev)), and apply it.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The PPV, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise G", answer)`. Return only the code in one block, with a short comment above each step.
Test: Check with an assert statement that ppv(0.85, 0.85, 0.5) returns 0.85.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise G", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
se, sp, prev = 0.85, 0.90, 0.05
answer = round(se * prev / (se * prev + (1 - sp) * (1 - prev)), 3)
teach.check("Exercise G", answer)

**Observe and reflect.** With class weights, the model finds about two thirds of the events, but only about one in ten of its alarms is a real event. Would such a model be useful to plan the follow-up visits of a heart failure clinic? Which threshold would you choose, and who should take part in that choice?

**Prompt challenge, Section 6: Two precision-recall curves.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `Xe_train`, `Xe_test`, `ye_train` and `ye_test`, the training and test sets of the event within 12 months (prevalence about 0.034), with the columns age, egfr, lvef, the logarithm of ntprobnp, af and diabetes.
Task: Fit a pipeline of StandardScaler and LogisticRegression(max_iter=1000) once without and once with `class_weight="balanced"`, and draw both precision-recall curves in one figure with a horizontal line at the prevalence.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure and the two average precisions. Return only the code in one block, with a short comment above each step.
Test: The two average precisions must match the AP column of the table of Section 6.
</pre>

</details>

**What to check.** The two curves almost overlap, with an average precision of about 0.10: Class weights move the threshold, not the ranking of the patients.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 7. Finding subgroups: k-means on the patients with heart failure

*Lecture: [Unsupervised learning: k-means and phenogroups](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#unsupervised-learning-k-means-and-phenogroups)*

The registry holds about 400 patients with heart failure with reduced ejection fraction (HFrEF) or preserved ejection fraction (HFpEF). Unsupervised learning looks for groups without any outcome. The *k-means* algorithm places $k$ centres, assigns each patient to the nearest one, moves each centre to the mean of its patients and repeats until nothing changes. The course hid a phenogroup in every patient with heart failure. The clustering does not see it, and the end of the section compares the clusters with it.

**From the formulas to the code.** k-means minimises the within-cluster sum of squares $WCSS = \sum_{k}\sum_{i \in C_k} \lVert \mathbf{x}_i - \mu_k \rVert^2$, which `KMeans` reports as `inertia_`. The silhouette of a patient is $s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$, with $a(i)$ the mean distance to its own cluster and $b(i)$ to the nearest other cluster. `silhouette_score` returns the mean over all patients.

**Started here?** Run the catch-up cell below once. It runs the code that Section 7 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(7)

In [ ]:
# Standardised features of the patients with heart failure
hf = registry[registry["main_diagnosis"].isin(["HFrEF", "HFpEF"])].copy()
HF_FEATURES = ["age", "bmi", "egfr", "lvef", "ntprobnp", "lavi", "e_e_prime", "lv_mass_index", "tapse_mm"]
X_hf = StandardScaler().fit_transform(hf[HF_FEATURES].assign(ntprobnp=np.log(hf["ntprobnp"])))
print(len(hf), "patients with heart failure")

ks = range(2, 7)
wcss, sil = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
    wcss.append(km.inertia_)
    sil.append(silhouette_score(X_hf, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(list(ks), wcss, "o-", color="#1F5F8B")
ax[0].set_xlabel("k")
ax[0].set_ylabel("WCSS")
ax[1].plot(list(ks), sil, "o-", color="#9B1B1B")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Mean silhouette")
plt.show()
print(pd.DataFrame({"k": list(ks), "WCSS": np.round(wcss, 1), "silhouette": np.round(sil, 3)}).to_string(index=False))

**Python note.** `.isin([...])` keeps the rows whose value is one of the listed values. `range(2, 7)` gives the numbers 2 to 6, and the loop fits one k-means model for each. `n_init=10` starts each model from ten random positions of the centres and keeps the best result, because k-means can get stuck in a poor solution.

In [ ]:
# Three clusters, drawn on the first two principal components
km3 = KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE).fit(X_hf)
hf["cluster"] = km3.labels_
pca = PCA(n_components=2).fit(X_hf)
pcs = pca.transform(X_hf)
fig, ax = plt.subplots(figsize=(5.6, 4.6))
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    ax.scatter(pcs[hf["cluster"] == c, 0], pcs[hf["cluster"] == c, 1], s=14, color=color, label=f"cluster {c}")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.0f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.0f}%)")
ax.legend()
plt.show()
hf.groupby("cluster")[HF_FEATURES + ["diabetes", "af"]].mean().round(1).T

**Python note.** Principal component analysis (PCA) finds the directions along which the patients differ most. `explained_variance_ratio_` gives the share of the total variance that each direction carries. The plot only shows the clusters, it does not make them: k-means worked on all nine variables.

**Exercise H.** What is the mean silhouette of the k-means solution with three clusters on `X_hf`? Use `silhouette_score(X_hf, km3.labels_)` and round to three decimals.

<details><summary><b>Prompt route</b> for Exercise H: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `X_hf`, a NumPy array with one row per patient with heart failure and nine columns, such as age (years), lvef (%) and tapse_mm (mm), standardised to z-scores.
Task: Fit `KMeans(n_clusters=3, n_init=10, random_state=RANDOM_STATE)` to `X_hf` and compute the mean silhouette of its labels with `silhouette_score`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The mean silhouette, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise H", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print the number of patients in each cluster; the three numbers must add up to 407, the number of rows of `X_hf`.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise H", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
answer = round(silhouette_score(X_hf, km3.labels_), 3)
teach.check("Exercise H", answer)

In [ ]:
# The hidden truth of the clinic: the phenogroups
print(pd.crosstab(hf["cluster"], hf["hf_phenogroup"]))
print("Adjusted Rand index:", round(adjusted_rand_score(hf["hf_phenogroup"], hf["cluster"]), 3))

**Observe and reflect.** The mean silhouette is slightly higher for two clusters than for three, yet three clusters match the hidden phenogroups well. Which criterion should decide the number of phenogroups in a real study, where nothing is hidden to compare with? Name the clusters by their profile in the table.

**Prompt challenge, Section 7: A heat map of the clusters.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `hf`, the DataFrame of the patients with heart failure with a column cluster (0, 1 or 2), and `HF_FEATURES`, the list of the nine clustering variables.
Task: Compute the mean of each variable in each cluster, turn each variable into a z-score across the three clusters and draw the table as a heat map with the values written in the cells.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: One figure. Return only the code in one block, with a short comment above each step.
Test: The cluster with the lowest mean lvef must also have the highest mean ntprobnp.
</pre>

</details>

**What to check.** One cluster has a low ejection fraction and the highest NT-proBNP (HFrEF), one is the oldest with a large left atrium and a low TAPSE, and one is the youngest with the highest BMI and eGFR.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 8. Survival analysis: Kaplan-Meier and the log-rank test

*Lecture: [Survival analysis: Kaplan-Meier and the log-rank test](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#survival-analysis-kaplan-meier-and-the-log-rank-test)*

Follow-up ends for different patients at different times. A patient still free of events at the last visit is *censored*: The event may come later, but it was not seen. The Kaplan-Meier estimator uses every patient for as long as the patient was followed. The log-rank test asks whether two or more curves differ more than chance would explain.

**From the formulas to the code.** At each time $t_i$ with an event, $n_i$ patients are at risk and $d_i$ have the event. The estimate is $S(t) = \prod_{t_i \le t} \left(1 - \frac{d_i}{n_i}\right)$. The log-rank statistic is $X^2 = \sum_g \frac{(O_g - E_g)^2}{E_g}$, with $O_g$ the observed events of group $g$ and $E_g = \sum_i d_i\, n_{gi}/n_i$ the expected ones. With $G$ groups it is compared with a chi-square distribution with $G - 1$ degrees of freedom.

**Started here?** Run the catch-up cell below once. It runs the code that Section 8 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(8)

In [ ]:
# Kaplan-Meier by hand
def kaplan_meier(time, event):
    """Times of the steps and survival after each step, starting at (0, 1)."""
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)
    times, surv, s = [0.0], [1.0], 1.0
    for t in np.unique(time[event == 1]):
        at_risk = np.sum(time >= t)
        events = np.sum((time == t) & (event == 1))
        s = s * (1 - events / at_risk)
        times.append(t)
        surv.append(s)
    return np.array(times), np.array(surv)


def survival_at(times, surv, t):
    """S(t) read from the step function."""
    return surv[np.searchsorted(times, t, side="right") - 1]


t10 = [2, 3, 3, 5, 6, 8, 9, 11, 12, 12]
e10 = [1, 1, 0, 1, 0, 1, 0, 1, 0, 0]
tt, ss = kaplan_meier(t10, e10)
pd.DataFrame({"month": tt, "S(t)": ss.round(4)})

**Python note.** `np.unique` returns the sorted distinct times with an event. `np.sum(time >= t)` counts the patients still followed at time `t`, which is $n_i$, and the condition `(time == t) & (event == 1)` counts the events at that time, $d_i$. `np.searchsorted` finds where a time falls among the steps of the curve.

In [ ]:
# Kaplan-Meier curves by cluster and by the type of heart failure
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9), sharey=True)
for c, color in zip(range(3), ["#9B1B1B", "#1F5F8B", "#B97813"]):
    d = hf[hf["cluster"] == c]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[0].step(tt, ss, where="post", color=color, label=f"cluster {c} (n={len(d)})")
for kind, color in zip(["HFrEF", "HFpEF"], ["#9B1B1B", "#1F5F8B"]):
    d = hf[hf["main_diagnosis"] == kind]
    tt, ss = kaplan_meier(d["followup_months"], d["event"])
    ax[1].step(tt, ss, where="post", color=color, label=f"{kind} (n={len(d)})")
    print(kind, "S(24) =", round(survival_at(tt, ss, 24), 3))
for a in ax:
    a.set_xlabel("Months")
    a.legend(loc="lower left")
ax[0].set_ylabel("Free of death or hospitalisation")
plt.show()


def logrank(time, event, group):
    """Log-rank test for two or more groups: chi-square and p value."""
    time, event, group = np.asarray(time, float), np.asarray(event, int), np.asarray(group)
    labels = np.unique(group)
    observed = np.zeros(len(labels))
    expected = np.zeros(len(labels))
    for t in np.unique(time[event == 1]):
        at_risk = time >= t
        d = np.sum((time == t) & (event == 1))
        n = np.sum(at_risk)
        for j, g in enumerate(labels):
            observed[j] += np.sum((time == t) & (event == 1) & (group == g))
            expected[j] += d * np.sum(at_risk & (group == g)) / n
    chi2 = np.sum((observed - expected) ** 2 / expected)
    return chi2, stats.chi2.sf(chi2, len(labels) - 1)


for name, col in [("HFrEF vs HFpEF", "main_diagnosis"), ("clusters", "cluster")]:
    chi2, p = logrank(hf["followup_months"], hf["event"], hf[col])
    print(f"{name}: X2 = {chi2:.2f}, p = {p:.2g}")

**Python note.** `ax[0].step(..., where="post")` draws a step function that keeps each value until the next time, the way a Kaplan-Meier curve looks. `stats.chi2.sf(x, df)` from SciPy gives the probability of a chi-square value at least as large as `x`, which is the p value. `f"{p:.2g}"` prints a number with two significant digits.

**Exercise I.** What is the Kaplan-Meier estimate of being free of death or hospitalisation at 24 months for the patients with HFrEF? Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise I: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `hf`, a DataFrame of the patients with heart failure with the columns main_diagnosis ("HFrEF" or "HFpEF"), followup_months (months) and event (1 for death or hospitalisation, 0 for censored).
Task: Write a function `kaplan_meier(time, event)` that returns the times of the steps and the survival after each step, with S(t) = product of (1 - d_i / n_i) over the event times up to t. Apply it to the patients with HFrEF and read S at 24 months.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: S(24), rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise I", answer)`. Return only the code in one block, with a short comment above each step.
Test: For the times 2, 3, 3, 5, 6, 8, 9, 11, 12, 12 months with the events 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, the survival after month 11 must be 0.3657 after rounding to four decimals.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise I", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
d = hf[hf["main_diagnosis"] == "HFrEF"]
tt, ss = kaplan_meier(d["followup_months"], d["event"])
answer = round(survival_at(tt, ss, 24), 3)
teach.check("Exercise I", answer)

**Observe and reflect.** HFrEF and HFpEF do not differ much in this registry, while the three clusters do. What does this say about grouping patients by the ejection fraction alone? Which cluster would you follow more closely, and what else would you want to know before changing its care?

**Advanced path.** If the package `lifelines` is installed, compare the two functions of this section with `KaplanMeierFitter` and `multivariate_logrank_test` of `lifelines`. The log-rank statistic of `lifelines` uses the exact variance and is therefore slightly larger.

In [ ]:
# Advanced path
try:
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import multivariate_logrank_test
    d = hf[hf["main_diagnosis"] == "HFrEF"]
    kmf = KaplanMeierFitter().fit(d["followup_months"], d["event"])
    print("lifelines S(24):", round(float(kmf.survival_function_at_times(24).iloc[0]), 3))
    print(multivariate_logrank_test(hf["followup_months"], hf["cluster"], hf["event"]).summary)
except ImportError:
    print("lifelines is not installed; the functions of this section are enough for the course.")

**Prompt challenge, Section 8: Numbers at risk.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: `hf`, the DataFrame of the patients with heart failure with the columns cluster, followup_months and event.
Task: Build the table of the numbers at risk, the patients still followed and free of events, at 0, 12, 24, 36 and 48 months for each cluster, as printed under a Kaplan-Meier figure in a paper.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: A pandas DataFrame with one row per cluster. Return only the code in one block, with a short comment above each step.
Test: At month 0 the numbers must equal the sizes of the clusters.
</pre>

</details>

**What to check.** The numbers fall in every cluster, fastest in the cluster of the oldest patients.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 9. AI pair programming, part 2: Reviewing a model built by an assistant

*Lecture: [AI pair programming, part 2: Reviewing a model built by an assistant](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lecture.html#ai-pair-programming-part-2-reviewing-a-model-built-by-an-assistant)*

A participant asked an AI assistant for *a model that predicts death or hospitalisation within 12 months from the registry, with its AUC*. The code below is what came back. It runs without an error and reports an excellent result. Read it with the model review questions of the lecture before running it: Which variables may the model see, and on which patients is it measured?

**Started here?** Run the catch-up cell below once. It runs the code that Section 9 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(9)

In [ ]:
# Code from an AI assistant: predict the 12-month event and report the AUC
features = ["age", "egfr", "lvef", "ntprobnp", "af", "diabetes", "bmi", "sbp", "followup_months"]
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
rf.fit(registry[features], registry["event_12m"])
auc_reported = roc_auc_score(registry["event_12m"], rf.predict_proba(registry[features])[:, 1])
print("AUC:", round(auc_reported, 3))

**Observe and reflect.** The code has two problems. The first is a variable that the model could never know at the time of the decision. The second is where the AUC is measured. Name both, and say which question of the review catches each.

**Exercise J.** Correct the code: Leave `followup_months` out, use `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)` and report the mean AUC of a five-fold cross-validation with `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)`. Round to three decimals.

<details><summary><b>Prompt route</b> for Exercise J: A ready prompt for an AI assistant</summary>

Copy the prompt into an AI assistant, replace the content of the next cell with the code it returns and run the cell. The code ends with the line of the check, which prints *correct* or *not yet*. If it prints *not yet*, use the repair prompt of the [prompt library](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/en/week-02/PROMPTS.md#repair-code-that-does-not-pass-the-check).

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: A pandas DataFrame named `registry` holds 2000 synthetic patients of the virtual clinic, one row per patient, made with `vclinic.add_followup(vclinic.make_cohort(2000, seed=RANDOM_STATE))`. Among its columns are age (years), sex ("M" or "F"), smoker, diabetes, hypertension, family_history and cad (0 or 1), ldl and hdl (mg/dL), egfr (mL/min/1.73 m²), lvef (%), ntprobnp (pg/mL), af (0 or 1), bmi (kg/m²), sbp (mmHg), main_diagnosis, followup_months (months) and event (1 for death or hospitalisation for heart failure, 0 for censored). The column event_12m is 1 for an event within 12 months. The list `features` holds the columns age, egfr, lvef, ntprobnp, af, diabetes, bmi, sbp and followup_months.
Task: Remove followup_months from the variables, because it is not known at the time of the decision. Then compute the mean AUC of `RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)` for event_12m with a five-fold `StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)`.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: The mean cross-validated AUC, rounded to three decimals. Store it in a variable named `answer` and end the code with the line `teach.check("Exercise J", answer)`. Return only the code in one block, with a short comment above each step.
Test: Print also the AUC of the same forest fitted and scored on all patients; it must be clearly higher than the cross-validated value, above 0.95.
</pre>

</details>

In [ ]:
answer = None   # your answer
teach.check("Exercise J", answer)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
fixed = [f for f in features if f != "followup_months"]
forest_cv = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
answer = round(cross_val_score(forest_cv, registry[fixed], registry["event_12m"], cv=folds, scoring="roc_auc").mean(), 3)
teach.check("Exercise J", answer)

The [prompt workshop](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lab.html#part-f) of the interactive lab writes the prompt for such a task with the seven fields of the prompt card. The field *Test* is the place for the check that would have caught both problems: a test of the model on patients it has not seen.

**Prompt challenge, Section 9: A review of the assistant's model.** Copy the prompt into an AI assistant, paste the code it returns into the empty cell below and run it.

<details><summary><b>Prompt</b> for an AI assistant</summary>

<pre>
Role: You are helping a cardiologist who is learning Python.
Context: A Jupyter notebook in which NumPy is imported as np, pandas as pd, matplotlib.pyplot as plt and the module stats of SciPy as stats, and scikit-learn is installed. The course package vclinic with its module teach is available, and RANDOM_STATE = 2026.
Data: The code of the AI assistant in Section 9, pasted into the prompt as text.
Task: Review the code with four questions: Which variables would not be known at the time of the decision? Does any step learn from all patients before the split? On which patients is the AUC measured? Is the outcome rare? Then return the corrected code.
Constraints: Use only these packages and the variables named here. Do not load files and do not install anything.
Output: A short list of the problems and the corrected code. Return only the code in one block, with a short comment above each step.
Test: The corrected code must report a cross-validated AUC, not the AUC on the training patients.
</pre>

</details>

**What to check.** The review names the follow-up time as information from the future and the AUC measured on the training patients; the corrected AUC is about 0.85 instead of 1.0.

In [ ]:
# Prompt challenge: paste the code from the AI assistant here and run it.

## 10. Application key: Another outcome, other variables

Change `OUTCOME` and `VARIABLES` and run the cell again. It reports the cross-validated AUC of a logistic regression and of a random forest, and the odds ratios of the logistic regression. Outcomes worth a try: `"cad"`, `"event_12m"`, `"af"` and `"event"`. For each, think first about which variables would be known at the time of the decision.

**Started here?** Run the catch-up cell below once. It runs the code that Section 10 needs from the earlier sections, without their outputs.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
def _catch_up(upto, name="NB02_machine_learning_fundamentals.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/en/week-02/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Catch-up finished: the code of the earlier sections has run.")


_catch_up(10)

In [ ]:
OUTCOME = "af"
VARIABLES = ["age", "bmi", "hypertension", "lvef", "lavi", "ntprobnp"]

Xo = registry[VARIABLES].copy()
if "ntprobnp" in Xo:
    Xo["ntprobnp"] = np.log(Xo["ntprobnp"])
yo = registry[OUTCOME]
folds = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
lr_o = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
rf_o = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE)
print("Prevalence of", OUTCOME, round(yo.mean(), 3))
for name, m in [("logistic regression", lr_o), ("random forest", rf_o)]:
    s = cross_val_score(m, Xo, yo, cv=folds, scoring="roc_auc")
    print(f"{name}: AUC {s.mean():.3f} ± {s.std():.3f}")
lr_o.fit(Xo, yo)
pd.Series(np.exp(lr_o.named_steps["logisticregression"].coef_[0]), index=VARIABLES,
          name="odds ratio per SD").round(2)

## Application challenges by field

| Field | Challenge |
|---|---|
| Heart failure | Add the follow-up NT-proBNP of your own clinic's protocol as a thought experiment: Which of the nine clustering variables would you replace to make the phenogroups easier to use at the bedside? Cluster again with your choice and compare the Kaplan-Meier curves. |
| Interventional cardiology | Build a model of coronary artery disease from the registry with the variables of Section 5, choose the threshold at which it would send no more than 20 percent of the patients without disease to angiography, and report its sensitivity at that threshold. |
| Electrophysiology | Predict atrial fibrillation (`af`) from age, BMI, hypertension, left atrial volume index and NT-proBNP. Which variable carries most of the information, and does the random forest add anything to the logistic regression? |
| Cardiovascular imaging | Cluster the patients with heart failure with the echocardiographic variables only (`lvef`, `lavi`, `e_e_prime`, `lv_mass_index`, `tapse_mm`) and compare the agreement with the hidden phenogroups with that of the nine variables. |
| Preventive cardiology | Draw the predicted probability of coronary artery disease of `cad_model` against age for a smoker and a non-smoker with otherwise average values. How many years of age does smoking add in this model? |
| Cardio-oncology | Imagine a screening model for cardiotoxicity with a sensitivity of 0.90 and a specificity of 0.80. Plot its PPV for prevalences from 1 to 20 percent and decide at which prevalence it would be worth using. |
| Intensive care | Use the 12-month event model with class weights and choose the threshold that finds 80 percent of the events. How many alarms does it raise per 100 patients, and how many of them are real events? |

## After the notebook

Open the [interactive lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/en/week-02/lab.html) and take the self-assessment in the tab *Check yourself*. Then write the reflection in the tab *Reflect and export* and export the learning log. The weekly project of Week 2, described on the [week page](README.md), starts from the clusters of Section 7 and the two functions of Section 8.

<div align="center">

---

**Prof. Dr. Utku Kose**<br>
Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University<br>
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Artificial Intelligence with Python: A Hands-on Training Program for Cardiologists, Turkish Society of Cardiology. First version: November 2026. Content CC BY 4.0, code MIT.*

</div>